# Time-Efficient Deep Learning Model for Leaf and Fruit Disease Detection
### Academic Research & Experimental Pipeline Demonstration

**Objective**: Evaluate lightweight convolutional neural network architectures (**MobileNetV3-Small** vs. **EfficientNetB0**) and **TensorFlow Lite INT8 Quantization** to achieve an optimal trade-off between plant disease classification accuracy, inference latency, parameter count, and storage footprint.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is on path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import config
from src.utils import set_random_seeds, get_hardware_info, estimate_flops
from src.data_loader import validate_dataset_structure, load_dataset_pipelines
from src.model import build_disease_model, get_model_parameter_counts
from src.predict import DiseasePredictor

set_random_seeds(config.RANDOM_SEED)
print("TensorFlow Version:", tf.__version__)
print("Hardware Specs:", get_hardware_info())

## 1. Dataset Verification and Exploration

In [ ]:
summary = validate_dataset_structure(config)
print("Dataset Summary:")
for split, counts in summary.items():
    print(f"  Split '{split}': {sum(counts.values())} total images across {len(counts)} classes")

## 2. Model Architecture Comparison

In [ ]:
mobilenet_model = build_disease_model(model_name="mobilenet_v3_small", num_classes=4)
efficientnet_model = build_disease_model(model_name="efficientnet_b0", num_classes=4)

mb_params = get_model_parameter_counts(mobilenet_model)
eff_params = get_model_parameter_counts(efficientnet_model)

arch_comp = pd.DataFrame([
    {"Architecture": "MobileNetV3-Small", **mb_params, "FLOPs": estimate_flops(mobilenet_model)},
    {"Architecture": "EfficientNetB0", **eff_params, "FLOPs": estimate_flops(efficientnet_model)}
])
arch_comp

## 3. Experimental Benchmarks & Latency Profiling

In [ ]:
comparison_csv = config.RESULTS_PATH / "model_comparison.csv"
if comparison_csv.exists():
    df_bench = pd.read_csv(comparison_csv)
    display(df_bench)
else:
    print("Run 'python src/benchmark.py' to generate comprehensive empirical benchmarks.")

## 4. Single-Sample Diagnostic Prediction

In [ ]:
# Find a test sample image
test_images = list(config.TEST_DIR.glob("*/*.jpg")) + list(config.TEST_DIR.glob("*/*.png"))
if test_images:
    sample_path = test_images[0]
    print(f"Testing image: {sample_path}")
    try:
        predictor = DiseasePredictor()
        result = predictor.predict(sample_path)
        print(f"Predicted Disease : {result['predicted_disease']}")
        print(f"Confidence Score  : {result['confidence_percent']}%")
        print(f"Inference Latency : {result['inference_time_ms']} ms")
        print(f"Status            : {result['status_message']}")
    except Exception as err:
        print("Predictor requires a trained model:", err)
else:
    print("No test images found.")